# Live Tracing and Native Desktop GUI

Graphyco provides a native desktop application using PySide6 that can connect directly to your PyTorch training loop to visualize live telemetry metrics, active bottlenecks, and gradient health in real-time.

## Setting up Live Diagnostics

For the real-time PySide6 GUI, we use `LiveTrainingDiagnostics`. This module hooks directly into your model's forward and backward passes to capture tensor statistics without slowing down training significantly.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from graphyco import LiveTrainingDiagnostics

# Define your architecture
model = nn.Sequential(
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 10)
)

# Initialize optimizer & loss criterion
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Initialize live diagnostics
diagnostics = LiveTrainingDiagnostics(model)
diagnostics.dummy_input = torch.randn(1, 128)  # Required to build the initial graph topology in the GUI

## Launching the Real-Time GUI

You can launch the GUI and pass the `diagnostics` object to it. The GUI will start polling your training loop automatically.

> **Note:** In a Jupyter Notebook environment, launching a native GUI window will block the execution of subsequent cells until the window is closed, unless you integrate it using the `%gui qt` magic or run your training loop in a background thread. For standard Python scripts, simply run `app.exec()`.

In [ ]:
# 1. Enable non-blocking Qt event loop for Jupyter
%gui qt

import sys
import threading
import time
import torch
from PySide6.QtWidgets import QApplication
from PySide6.QtCore import QCoreApplication, Qt
from graphyco.visualizer.gui import launch_desktop_app

# Safe OpenGL sharing attribute check (must be set before QApplication creation if not already created)
if not QApplication.instance():
    QCoreApplication.setAttribute(Qt.ApplicationAttribute.AA_ShareOpenGLContexts)

app = QApplication.instance() or QApplication([])
window = launch_desktop_app(diagnostics=diagnostics)

# 2. Define a background training loop
def training_thread():
    print("Training loop started in background...")
    for step in range(100):
        x = torch.randn(16, 128)
        y = torch.randint(0, 10, (16,))
        with diagnostics.observe_step(step):
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            diagnostics.current_loss = loss.item()
            loss.backward()
            optimizer.step()
        time.sleep(0.1)  # Pace steps so the live GUI updates smoothly
    print("Training loop finished.")

# 3. Start training in the background
thread = threading.Thread(target=training_thread, daemon=True)
thread.start()

# 4. Skip app.exec() because %gui qt handles the event loop automatically in the background!
print("Graphyco Live Diagnostic GUI is now running in the background.")

## Alternative: Running via CLI

If you prefer to run the GUI from the terminal on an existing JSON benchmark export instead of a live model, you can run:

```bash
python -m graphyco.visualizer --json my_model_benchmark.json
```